# Data Loading

In [61]:
%%time

import kplex_dev
import data_splits
import warnings
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder


warnings.filterwarnings('ignore')

timeout_replacement = 1800  # max timeout replacement
excel_path = "kplex_ML_data.xlsx"


CPU times: total: 0 ns
Wall time: 2.97 ms


In [62]:
# Loading and preprocessing data
cat_df, dfs = kplex_dev.load_excel_data(excel_path, timeout_replacement)
kplex_dev.process_kcore_list_to_statistics(cat_df)

# Selecting weight scheme
epsilon = 0.001 # need to set !!!
scheme = kplex_dev.W_TIME_MIN_X_W_LOG_TIME_RATIO # Change this and analyse!!

# try dense cross entrpy loss for NN
# take 10% of data out as unseen data and run triaing and test split on remaining 90%data

# Assigning weights
kplex_dev.assign_weights(cat_df, scheme, epsilon)

# kplex_dev.export_sorted(cat_df)

# Data reformatting
cat_df = kplex_dev.post_process_data(cat_df)

# fastest_time = minimum across those algorithms
cat_df['fastest_time'] = cat_df[data_splits.algorithms].min(axis=1)
cat_df

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize,weight,UMKP,AltRB,kPEX,Pivot_Br,S_Br,fastest_time
0,MANN-a81,20,3321,5506380,1658.0488,3317,3280,3280,1,3321.000000,3321,3321,0.000000,306.046574,1459.5,1800.0,1468.1,1477.4,1504.7,1459.5
1,wikipedia_link_sr,20,1810,1637059,904.4525,1809,1798,1795,4,1806.250000,1810,1803,2.861381,66.902739,22.6,18.0,737.7,740.4,738.1,18.0
2,scc_reality,20,1332,880733,661.2110,1331,1235,1230,6,1290.333333,1332,1236,35.714921,47.684856,1800.0,1214.8,9.4,9.7,9.0,9.0
3,web-wikipedia_link_en13-all,20,432,93085,215.4745,431,427,427,1,432.000000,432,432,0.000000,184.662528,239.3,287.1,517.7,504.4,516.7,239.3
4,livejournal-links,20,442,94435,213.6538,441,371,369,3,412.333333,442,386,22.983086,0.269931,1.3,1.6,1.6,1.6,1.6,1.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,soc-flixster,2,264,11792,44.6667,150,67,52,16,215.000000,264,125,56.656862,0.784308,2.0,0.6,0.5,0.5,2.4,0.5
756,edit-ukwiki,2,281,14183,50.4733,280,65,40,26,235.384615,281,168,34.703222,0.437734,1.2,1.1,0.6,0.6,0.5,0.5
757,johnson8-4-4,2,70,1855,26.5000,53,53,53,1,70.000000,70,70,0.000000,0.389785,0.8,0.9,1.1,1.1,0.3,0.3
758,hamming6-2,2,64,1824,28.5000,57,57,57,1,64.000000,64,64,0.000000,0.081093,0.2,0.3,0.2,0.2,0.3,0.2


# Training with Neural Network

Simple neural network with a soft‑cross‑entropy loss.


### Model

In [63]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.metrics import classification_report

In [64]:
# --- 1) Soft Cross‐Entropy with Temperature --------------------------

class SoftCrossEntropyLoss(nn.Module):
    """
    Cross‐entropy loss where logits are divided by a temperature T before softmax.
    """
    def __init__(self, temperature: float = 1.0):
        super().__init__()
        self.temperature = temperature

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        # scale logits
        scaled = logits / self.temperature
        return F.cross_entropy(scaled, targets)

In [65]:
# kplex_dev.py

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted

# ——— 1) Soft Cross‐Entropy with Temperature ——————————————

class SoftCrossEntropyLoss(nn.Module):
    """
    Cross‐entropy loss where logits are divided by temperature T before softmax.
    """
    def __init__(self, temperature: float = 1.0):
        super().__init__()
        self.temperature = temperature

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        scaled = logits / self.temperature
        return F.cross_entropy(scaled, targets)


# ——— 2) sklearn‐style PyTorch MLP Classifier ——————————————————

class TorchNNClassifier(BaseEstimator, ClassifierMixin):
    """
    A sklearn‐compatible PyTorch MLP with 3–5 hidden layers, soft‐CE loss.
    """
    def __init__(
        self,
        hidden_layers=(128, 64, 32),  # tuple of 3–5 ints
        activation: str = 'relu',     # 'relu' | 'tanh'
        lr: float = 1e-2,
        batch_size: int = 64,
        epochs: int = 50,
        temperature: float = 1.0,
        device: str = None,
        random_state: int = 42
    ):
        self.hidden_layers = hidden_layers
        self.activation    = activation
        self.lr            = lr
        self.batch_size    = batch_size
        self.epochs        = epochs
        self.temperature   = temperature
        self.device        = device
        self.random_state  = random_state

    def _build_model(self, n_features, n_classes):
        layers = []
        in_dim = n_features
        act_fn = nn.ReLU if self.activation=='relu' else nn.Tanh
        # hidden layers
        for h in self.hidden_layers:
            layers.append(nn.Linear(in_dim, h))
            layers.append(act_fn())
            in_dim = h
        # output layer
        layers.append(nn.Linear(in_dim, n_classes))
        return nn.Sequential(*layers)

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        n_samples, n_features = X.shape
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)

        # reproducibility & device
        torch.manual_seed(self.random_state) # comment for checking randomness of algorithm
        self.device_ = torch.device(self.device or ('cuda' if torch.cuda.is_available() else 'cpu'))

        # scale features
        self.scaler_ = StandardScaler().fit(X)
        X_scaled   = self.scaler_.transform(X)

        # map classes to indices
        self.class_to_idx_ = {c:i for i,c in enumerate(self.classes_)}
        y_idx = np.vectorize(self.class_to_idx_.get)(y)

        # DataLoader
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=self.device_)
        y_tensor = torch.tensor(y_idx,    dtype=torch.long,   device=self.device_)
        ds = TensorDataset(X_tensor, y_tensor)
        loader = DataLoader(ds, batch_size=self.batch_size, shuffle=True)

        # model, loss, optimizer
        self.model_ = self._build_model(n_features, n_classes).to(self.device_)
        criterion = SoftCrossEntropyLoss(self.temperature)
        
        optimizer = optim.SGD(self.model_.parameters(), lr=self.lr, weight_decay=0)

        # training
        for epoch in range(1, self.epochs+1):
            self.model_.train()
            for xb, yb in loader:
                optimizer.zero_grad()
                logits = self.model_(xb)
                loss   = criterion(logits, yb)
                loss.backward()
                optimizer.step()

        return self

    def predict(self, X):
        check_is_fitted(self, ['model_','scaler_'])
        X = check_array(X)
        X_scaled = self.scaler_.transform(X)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=self.device_)
        self.model_.eval()
        with torch.no_grad():
            logits = self.model_(X_tensor)
            preds  = logits.argmax(dim=1).cpu().numpy()
        inv_map = {i:c for c,i in self.class_to_idx_.items()}
        return np.vectorize(inv_map.get)(preds)

    def predict_proba(self, X):
        check_is_fitted(self, ['model_','scaler_'])
        X = check_array(X)
        X_scaled = self.scaler_.transform(X)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=self.device_)
        self.model_.eval()
        with torch.no_grad():
            logits = self.model_(X_tensor)
            scaled = logits / self.temperature
            probs = F.softmax(scaled, dim=1).cpu().numpy()
        return probs


## Tie Breaking and Data Splitting

### **Balanced train-test split based on Y**

In [66]:
from data_splits import prepare_train_test_strat_Y
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y(cat_df, test_size=0.2,
    tie_tolerance=0.50  # algorithms within tie_tolerance% of min treated as ties
    )
# Label‐encode the string targets into integers 0…4 ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


Expanded 687 rows with near ties using tolerance 50.0%.
Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


#### NN with 3 layers

In [70]:
# Instantiate and train
clf_nn_strat_y = TorchNNClassifier(
    hidden_layers=(128, 64, 32),
    activation='relu',
    lr=0.5,
    batch_size=32,
    epochs=1000,
)
clf_nn_strat_y.fit(X_train, y_train)


TorchNNClassifier(batch_size=32, epochs=1000, lr=0.5)

Classification report and Feature Importances

In [71]:
y_pred_int = clf_nn_strat_y.predict(X_test)
y_pred_str = le.inverse_transform(y_pred_int)
print("\nClassification Report on Test Set:")
print(classification_report(y_test_str, y_pred_str))


Classification Report on Test Set:
              precision    recall  f1-score   support

       AltRB       0.17      0.21      0.19        52
    Pivot_Br       0.11      0.05      0.07       115
        S_Br       0.18      0.10      0.13        87
        UMKP       0.06      0.04      0.05        50
        kPEX       0.31      0.59      0.40       116

    accuracy                           0.23       420
   macro avg       0.17      0.20      0.17       420
weighted avg       0.18      0.23      0.19       420



#### Inference on 20% dataset

In [72]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=clf_nn_strat_y,
    le=le,
    for_nn = False, # set False if you don’t want to use fastest algorithm runtime as a feature
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./NN_results/tie_tol/50/expand_and_split/',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./NN_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 55.71%

Precision by Algorithm:
  UMKP: 29.03% (31 predictions)
  AltRB: 42.86% (63 predictions)
  kPEX: 63.96% (222 predictions)
  Pivot_Br: 50.91% (55 predictions)
  S_Br: 57.14% (49 predictions)

Recall by Algorithm:
  UMKP: 46.94% (23/49 cases)
  AltRB: 63.33% (38/60 cases)
  kPEX: 78.14% (168/215 cases)
  Pivot_Br: 65.16% (101/155 cases)
  S_Br: 55.03% (82/149 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 65701.00
  Total Optimal Time   = 48282.00
  Time Ratio (pred/opt)= 1.361
  Avg Overhead/case ((pred - opt) / opt)    = 1706.78%


Adding prediction probabilities...

Saving full predictions to './NN_results/tie_tol/50/expand_and_split/all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 186 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
             dataset  k_value 

#### Re-evalute on entire dataset

In [73]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_nn_strat_y,
    le=le,
    for_nn = False, # set False if you don’t want to use fastest algorithm runtime as a feature
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=False,               # set False if you don’t want to write files
    output_dir='./NN_results/tie_tol/50/expand_and_split/',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./NN_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 60.92%

Precision by Algorithm:
  UMKP: 41.46% (41 predictions)
  AltRB: 52.07% (121 predictions)
  kPEX: 65.26% (426 predictions)
  Pivot_Br: 46.75% (77 predictions)
  S_Br: 72.63% (95 predictions)

Recall by Algorithm:
  UMKP: 45.00% (45/100 cases)
  AltRB: 69.83% (81/116 cases)
  kPEX: 83.47% (303/363 cases)
  Pivot_Br: 61.50% (139/226 cases)
  S_Br: 60.94% (156/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 87848.90
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.314
  Avg Overhead/case ((pred - opt) / opt)    = 163.27%


Adding prediction probabilities...

Generating wrong-predictions with time metrics...

Saved 297 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
           dataset  k_value      weight predicted fastest_algorithms  \
646     bag-pubmed        3  662.18

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

In [ ]:
## SHowing row 425

rt_opt  = cat_df[data_splits.algorithms].min(axis=1)
cat_df[rt_opt==0]

#### Check for randomness
Running user-specifed number of experiments.

Number of experiments = 5

In [ ]:
# for i in range(1, 5 + 1):
#     exp_code = f"{i:02d}"

#     print(f"\n=== Experiment {exp_code} / 05: Initializing model ===")
#     # 1) Instantiate & train a fresh classifier each run
#     print("-> Creating TorchNNClassifier with temperature=0.5, lr=0.15, epochs=1000...")
#     clf_nn_strat_y = TorchNNClassifier(
#         hidden_layers=(128, 64, 32),
#         activation='relu',
#         lr=0.15,
#         batch_size=32,
#         epochs=1000,
#         temperature=1.0
#     )
#     clf_nn_strat_y.fit(X_train, y_train)
#     print("✔ Training complete.")

#     # 2) Evaluate on test hold‑out set
#     print(f"\n--- Experiment {exp_code}: Evaluating on test set ---")

#     classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
#     cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
#     clf=clf_nn_strat_y,
#     le=le,
#     features=data_splits.feature_cols,
#     algorithms=data_splits.algorithms,
#     # drop_indices= [425],
#     epsilon=1e-3,
#     save_files=True,               # set False if you don’t want to write files
#     output_dir='./NN_results/tie_tol/50/expand_and_split/strat_y',
#     file_suffix= f"inference_test_set_without_holdout_strat_y_exp_{exp_code}" # change exp number
#     )

#     print("✔ Test‑set evaluation complete. Metrics and predictions saved.")

#     # 3) Write test metrics into performance workbook
#     print("-> Appending test‑set metrics to Excel workbook...")

#     # Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
#     kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
#                                         "./NN_results/tie_tol/50/expand_and_split/strat_y/model_performance_for_all_experiments.xlsx", 
#                                         f'Inference_test_set_exp_{exp_code}') # change exp number
#     print("✔ Test‑set metrics written.")

#     # 4) Evaluate on entire dataset
#     print(f"\n--- Experiment {exp_code}: Evaluating on entire dataset ---")
    
#     classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
#         cat_df=cat_df,
#         clf=clf_nn_strat_y,
#         le=le,
#         features=data_splits.feature_cols,
#         algorithms=data_splits.algorithms,
#         drop_indices= [425],
#         epsilon=1e-3,
#         save_files=True,               # set False if you don’t want to write files
#         output_dir='./NN_results/tie_tol/50/expand_and_split/strat_y',
#         file_suffix= f"inference_entire_data_without_holdout_strat_y_exp_{exp_code}" # change exp number
#     )
#     print("✔ Entire‑data evaluation complete. Metrics and predictions saved.")

#     # 5) Write full‑data metrics into performance workbook
#     print("-> Appending entire‑data metrics to Excel workbook...")

#     # Helper function: Write the displayed metrics to an excel file.
#     kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
#                                         "./NN_results/tie_tol/50/expand_and_split/strat_y/model_performance_for_all_experiments.xlsx", 
#                                         f'Inference_entire_data_exp_{exp_code}') # change exp num
#     print("✔ Entire‑data metrics written.")

# print("\n=== All experiments complete ===")

### **Balanced train-test split based on both Y and k**

In [74]:
from data_splits import prepare_train_test_strat_Y_K
from sklearn.preprocessing import LabelEncoder

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y_K(cat_df, test_size=0.2,
    tie_tolerance=0.50  # algorithms within tie_tolerance% of min treated as ties
    )

# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")



Expanded 687 rows with near ties using tolerance 50.0%.
Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


#### NN with 3 layers

In [94]:
# Instantiate and train
clf_nn_strat_y_K = TorchNNClassifier(
    hidden_layers=(128, 64, 32),
    activation='relu',
    lr=0.25,
    batch_size=32,
    epochs=1000,
)

clf_nn_strat_y_K.fit(X_train, y_train)

y_pred_int = clf_nn_strat_y_K.predict(X_test)
y_pred_str = le.inverse_transform(y_pred_int)
print("\nClassification Report on Test Set:")
print(classification_report(y_test_str, y_pred_str, sample_weight=weights_test))


Classification Report on Test Set:
              precision    recall  f1-score   support

       AltRB       0.01      0.00      0.00 1483.3530488672318
    Pivot_Br       0.21      0.37      0.27 3582.073469039411
        S_Br       0.47      0.50      0.49 1849.663881911921
        UMKP       0.13      0.17      0.15 786.182176800659
        kPEX       0.04      0.02      0.03 4679.182576808955

    accuracy                           0.20 12380.455153428178
   macro avg       0.17      0.21      0.19 12380.455153428178
weighted avg       0.16      0.20      0.17 12380.455153428178



#### Inference on 20% dataset

In [95]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=clf_nn_strat_y_K,
    le=le,
    for_nn = False, # set False if you don’t want to use fastest algorithm runtime as a feature
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./NN_results/tie_tol/50/expand_and_split/',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./NN_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 49.52%

Precision by Algorithm:
  UMKP: 44.00% (25 predictions)
  AltRB: 50.00% (42 predictions)
  kPEX: 63.51% (74 predictions)
  Pivot_Br: 43.88% (196 predictions)
  S_Br: 51.81% (83 predictions)

Recall by Algorithm:
  UMKP: 67.92% (36/53 cases)
  AltRB: 59.42% (41/69 cases)
  kPEX: 51.57% (115/223 cases)
  Pivot_Br: 81.29% (126/155 cases)
  S_Br: 63.01% (92/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 58302.70
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.256
  Avg Overhead/case ((pred - opt) / opt)    = 90.50%


Adding prediction probabilities...

Saving full predictions to './NN_results/tie_tol/50/expand_and_split/all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 212 wrong predictions with time metrics

Top 5 heav

#### Re-evalute on entire dataset

In [93]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_nn_strat_y_K,
    le=le,
    for_nn = False, # set False if you don’t want to use fastest algorithm runtime as a feature
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./NN_results/tie_tol/50/expand_and_split/',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./NN_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...



Classification performance:
Accuracy (counting ties as correct): 48.82%

Precision by Algorithm:
  UMKP: 42.37% (59 predictions)
  AltRB: 60.00% (85 predictions)
  kPEX: 58.58% (169 predictions)
  Pivot_Br: 29.82% (218 predictions)
  S_Br: 57.21% (229 predictions)

Recall by Algorithm:
  UMKP: 57.00% (57/100 cases)
  AltRB: 59.48% (69/116 cases)
  kPEX: 49.86% (181/363 cases)
  Pivot_Br: 64.60% (146/226 cases)
  S_Br: 71.48% (183/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 88057.90
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.317
  Avg Overhead/case ((pred - opt) / opt)    = 547.41%


Adding prediction probabilities...

Saving full predictions to './NN_results/tie_tol/50/expand_and_split/all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 389 wrong predictions with time metrics

Top 5 heaviest wrong prediction

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

In [ ]:
## SHowing row 425

rt_opt  = cat_df[data_splits.algorithms].min(axis=1)
cat_df[rt_opt==0]

#### Check for randomness
Running user-specifed number of experiments.

Number of experiments = 5

In [ ]:
# for i in range(1, 5 + 1):
#     exp_code = f"{i:02d}"

#     print(f"\n=== Experiment {exp_code} / 05: Initializing model ===")
#     # 1) Instantiate & train a fresh classifier each run
#     print("-> Creating TorchNNClassifier with temperature=0.5, lr=0.15, epochs=1000...")

#     clf_nn_strat_y_K = TorchNNClassifier(
#         hidden_layers=(128, 64, 32),
#         activation='relu',
#         lr=0.15,
#         batch_size=32,
#         # batch_size=X_train.shape[0], 
#         epochs=1000,
#         temperature=1.0
#     )
#     clf_nn_strat_y_K.fit(X_train, y_train)

#     print("✔ Training complete.")

#     # 2) Evaluate on test hold‑out set
#     print(f"\n--- Experiment {exp_code}: Evaluating on test set ---")

#     classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
#     cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
#     clf=clf_nn_strat_y_K,
#     le=le,
#     features=data_splits.feature_cols,
#     algorithms=data_splits.algorithms,
#     drop_indices= [425],
#     epsilon=1e-3,
#     save_files=True,               # set False if you don’t want to write files
#     output_dir='./NN_results/tie_tol/50/expand_and_split/strat_y_K',
#     file_suffix= f"inference_test_set_without_holdout_strat_y_K_exp_{exp_code}" # change exp number
#     )

#     print("✔ Test‑set evaluation complete. Metrics and predictions saved.")

#     # 3) Write test metrics into performance workbook
#     print("-> Appending test‑set metrics to Excel workbook...")


#     # Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
#     kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
#                                         "./NN_results/tie_tol/50/expand_and_split/strat_y_K/model_performance_for_all_experiments.xlsx", 
#                                         f'Inference_test_set_exp_{exp_code}') # change exp number

#     print("✔ Test‑set metrics written.")

#     # 4) Evaluate on entire dataset
#     print(f"\n--- Experiment {exp_code}: Evaluating on entire dataset ---")
    
#     classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
#         cat_df=cat_df,
#         clf=clf_nn_strat_y_K,
#         le=le,
#         features=data_splits.feature_cols,
#         algorithms=data_splits.algorithms,
#         drop_indices= [425],
#         epsilon=1e-3,
#         save_files=True,               # set False if you don’t want to write files
#         output_dir='./NN_results/tie_tol/50/expand_and_split/strat_y_K',
#         file_suffix= f"inference_entire_data_without_holdout_strat_y_K_exp_{exp_code}" # change exp number
#     )
#     print("✔ Entire‑data evaluation complete. Metrics and predictions saved.")

#     # 5) Write full‑data metrics into performance workbook
#     print("-> Appending entire‑data metrics to Excel workbook...")

#     # Helper function: Write the displayed metrics to an excel file.
#     kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
#                                         "./NN_results/tie_tol/50/expand_and_split/strat_y_K/model_performance_for_all_experiments.xlsx", 
#                                         f'Inference_entire_data_exp_{exp_code}') # change exp number
#     print("✔ Entire‑data metrics written.")

# print("\n=== All experiments complete ===")

## Data Splitting and Tie Breaking 

First we split the data and then do the tie breaking.

Only use training set to produce multi-label aware training set with duplicate lines (but different labels)


### **Balanced train-test split based on Y**

In [ ]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    test_size=0.2,
    random_state=40,
    # how='first'
)

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target',
    tie_tolerance=0.50  # algorithms within tie_tolerance% of min treated as ties
    for_nn = False, # set False if you don’t want to use fastest algorithm runtime as a feature
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


In [ ]:
# Instantiate & train NN with 3 layers
clf_nn_holdout_strat_y = TorchNNClassifier(
    hidden_layers=(128, 64, 32),
    activation='relu',
    lr=0.05,
    batch_size=32,
    epochs=1000,
    temperature=1.0
)

clf_nn_holdout_strat_y.fit(X_train_, y_train_)

#### Inference on 20% dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_nn_holdout_strat_y,
    le=le,
    for_nn = False, # set False if you don’t want to use fastest algorithm runtime as a feature
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./NN_results/tie_tol/50/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./NN_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

#### Re-evalute on entire dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_nn_holdout_strat_y,
    le=le,
    for_nn = False, # set False if you don’t want to use fastest algorithm runtime as a feature
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./NN_results/tie_tol/50/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./NN_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

#### Check for randomness
Running user-specifed number of experiments.

Number of experiments = 5

In [ ]:
# for i in range(1, 5 + 1):
#     exp_code = f"{i:02d}"

#     print(f"\n=== Experiment {exp_code} / 05: Initializing model ===")
#     # 1) Instantiate & train a fresh classifier each run
#     print("-> Creating TorchNNClassifier with temperature=0.5, lr=0.15, epochs=1000...")
#     clf_nn_holdout_strat_y = TorchNNClassifier(
#         hidden_layers=(128, 64, 32),
#         activation='relu',
#         lr=0.05,
#         batch_size=32,
#         epochs=1000,
#         temperature=1.0
#     )

#     clf_nn_holdout_strat_y.fit(X_train_, y_train_)
#     print("✔ Training complete.")

#     # 2) Evaluate on test hold‑out set
#     print(f"\n--- Experiment {exp_code}: Evaluating on test set ---")

#     classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
#         cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
#         clf=clf_nn_holdout_strat_y,
#         le=le,
#         features=data_splits.feature_cols,
#         algorithms=data_splits.algorithms,
#         # drop_indices= [425],
#         epsilon=1e-3,
#         save_files=True,               # set False if you don’t want to write files
#         output_dir='./NN_results/tie_tol/50/split_and_expand/strat_y',
#         file_suffix= f"inference_test_set_without_holdout_strat_y_exp_{exp_code}" # change exp number
#     )

#     print("✔ Test‑set evaluation complete. Metrics and predictions saved.")

#     # 3) Write test metrics into performance workbook
#     print("-> Appending test‑set metrics to Excel workbook...")

#     # Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
#     kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
#                                         "./NN_results/tie_tol/50/split_and_expand/strat_y/model_performance_for_all_experiments.xlsx", 
#                                         f'Inference_test_set_exp_{exp_code}') # change exp number
#     print("✔ Test‑set metrics written.")

#     # 4) Evaluate on entire dataset
#     print(f"\n--- Experiment {exp_code}: Evaluating on entire dataset ---")
    
#     classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
#         cat_df=cat_df,
#         clf=clf_nn_holdout_strat_y,
#         le=le,
#         features=data_splits.feature_cols,
#         algorithms=data_splits.algorithms,
#         drop_indices= [425],
#         epsilon=1e-3,
#         save_files=True,               # set False if you don’t want to write files
#         output_dir='./NN_results/tie_tol/50/split_and_expand/strat_y',
#         file_suffix= f"inference_entire_data_without_holdout_strat_y_exp_{exp_code}" # change exp number
#     )
#     print("✔ Entire‑data evaluation complete. Metrics and predictions saved.")

#     # 5) Write full‑data metrics into performance workbook
#     print("-> Appending entire‑data metrics to Excel workbook...")

#     # Helper function: Write the displayed metrics to an excel file.
#     kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
#                                     "./NN_results/tie_tol/50/split_and_expand/strat_y/model_performance_for_all_experiments.xlsx", 
#                                         f'Inference_entire_data_exp_{exp_code}') # change exp num
#     print("✔ Entire‑data metrics written.")

# print("\n=== All experiments complete ===")

### **Balanced train-test split based on Y and K**

In [ ]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    k_value_col='k_value',
    test_size=0.2,
    random_state=40,
    # how='first'
)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target', 
    for_nn = False, # set False if you don’t want to use fastest algorithm runtime as a feature
    tie_tolerance=0.50  # algorithms within tie_tolerance% of min treated as ties
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


In [ ]:
# Instantiate & train NN with 3 layers
clf_nn_holdout_strat_y_K = TorchNNClassifier(
    hidden_layers=(128, 64, 32),
    # hidden_layers=(8, 4),
    activation='relu',
    lr=0.01,
    batch_size=32,
    epochs=1000,
    temperature=1.0
)

clf_nn_holdout_strat_y_K.fit(X_train_, y_train_)

#### Inference on 20% dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_nn_holdout_strat_y_K,
    le=le,
    for_nn = False, # set False if you don’t want to use fastest algorithm runtime as a feature
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./NN_results/tie_tol/50/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./NN_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

#### Re-evalute on entire dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_nn_holdout_strat_y_K,
    le=le,
    for_nn = False, # set False if you don’t want to use fastest algorithm runtime as a feature
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./NN_results/tie_tol/50/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./NN_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

#### Check for randomness
Running user-specifed number of experiments.

Number of experiments = 5

In [ ]:
# for i in range(1, 5 + 1):
#     exp_code = f"{i:02d}"

#     print(f"\n=== Experiment {exp_code} / 05: Initializing model ===")
#     # 1) Instantiate & train a fresh classifier each run
#     print("-> Creating TorchNNClassifier with temperature=0.5, lr=0.15, epochs=1000...")

#     clf_nn_holdout_strat_y_K = TorchNNClassifier(
#         hidden_layers=(128, 64, 32),
#         # hidden_layers=(8, 4),
#         activation='relu',
#         lr=0.01,
#         batch_size=32,
#         epochs=1000,
#         temperature=1.0
#     )

#     clf_nn_holdout_strat_y_K.fit(X_train_, y_train_)

#     print("✔ Training complete.")

#     # 2) Evaluate on test hold‑out set
#     print(f"\n--- Experiment {exp_code}: Evaluating on test set ---")

#     classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
#         cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
#         clf=clf_nn_holdout_strat_y_K,
#         le=le,
#         features=data_splits.feature_cols,
#         algorithms=data_splits.algorithms,
#         drop_indices= [425],
#         epsilon=1e-3,
#         save_files=True,               # set False if you don’t want to write files
#         output_dir='./NN_results/tie_tol/50/split_and_expand/strat_y_K',
#         file_suffix= f"inference_test_set_without_holdout_strat_y_K_exp_{exp_code}" # change exp number
#     )

#     print("✔ Test‑set evaluation complete. Metrics and predictions saved.")

#     # 3) Write test metrics into performance workbook
#     print("-> Appending test‑set metrics to Excel workbook...")


#     # Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
#     kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
#                                     "./NN_results/tie_tol/50/split_and_expand/strat_y_K/model_performance_for_all_experiments.xlsx", 
#                                         f'Inference_test_set_exp_{exp_code}') # change exp number

#     print("✔ Test‑set metrics written.")

#     # 4) Evaluate on entire dataset
#     print(f"\n--- Experiment {exp_code}: Evaluating on entire dataset ---")
    
#     classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
#         cat_df=cat_df,
#         clf=clf_nn_holdout_strat_y_K,
#         le=le,
#         features=data_splits.feature_cols,
#         algorithms=data_splits.algorithms,
#         drop_indices= [425],
#         epsilon=1e-3,
#         save_files=True,               # set False if you don’t want to write files
#         output_dir='./NN_results/tie_tol/50/split_and_expand/strat_y_K',
#         file_suffix= f"inference_entire_data_without_holdout_strat_y_K_exp_{exp_code}" # change exp number
#     )
#     print("✔ Entire‑data evaluation complete. Metrics and predictions saved.")

#     # 5) Write full‑data metrics into performance workbook
#     print("-> Appending entire‑data metrics to Excel workbook...")

#     # Helper function: Write the displayed metrics to an excel file.
#     kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
#                                     "./NN_results/tie_tol/50/split_and_expand/strat_y_K/model_performance_for_all_experiments.xlsx", 
#                                         f'Inference_entire_data_exp_{exp_code}') # change exp number
#     print("✔ Entire‑data metrics written.")

# print("\n=== All experiments complete ===")